In [81]:
import csv
import os
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import sys
import os

# Go up one level from R&D and add HP_model to Python path
# sys.path.append(os.path.abspath('../HP_model'))

# Now you can import the classes directly
from Hydraplus import HP
from Mnet import MNet
from AF import AF

import torch
import torch.nn as nn
import torch.optim as optim
import sys
import os

In [82]:
import scipy.io

# Path to your .mat file
path = '/media/tanmoy002/HDD/Pedestrian Attribute Analysis/PA-100K/annotation.mat'

# Load the .mat file
mat_data = scipy.io.loadmat(path)

# Check keys in the loaded .mat file
print(mat_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'attributes', 'test_images_name', 'test_label', 'train_images_name', 'train_label', 'val_images_name', 'val_label'])


In [83]:
mat_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Wed Jul 26 13:11:29 2017',
 '__version__': '1.0',
 '__globals__': [],
 'attributes': array([[array(['Female'], dtype='<U6')],
        [array(['AgeOver60'], dtype='<U9')],
        [array(['Age18-60'], dtype='<U8')],
        [array(['AgeLess18'], dtype='<U9')],
        [array(['Front'], dtype='<U5')],
        [array(['Side'], dtype='<U4')],
        [array(['Back'], dtype='<U4')],
        [array(['Hat'], dtype='<U3')],
        [array(['Glasses'], dtype='<U7')],
        [array(['HandBag'], dtype='<U7')],
        [array(['ShoulderBag'], dtype='<U11')],
        [array(['Backpack'], dtype='<U8')],
        [array(['HoldObjectsInFront'], dtype='<U18')],
        [array(['ShortSleeve'], dtype='<U11')],
        [array(['LongSleeve'], dtype='<U10')],
        [array(['UpperStride'], dtype='<U11')],
        [array(['UpperLogo'], dtype='<U9')],
        [array(['UpperPlaid'], dtype='<U10')],
        [array(['UpperSplice'], dtype='<U11

In [84]:
# Extract the required data
attributes = [attr[0][0] for attr in mat_data['attributes']]
train_images = [img[0][0] for img in mat_data['train_images_name']]
train_labels = mat_data['train_label']
val_images = [img[0][0] for img in mat_data['val_images_name']]
val_labels = mat_data['val_label']
test_images = [img[0][0] for img in mat_data['test_images_name']]
test_labels = mat_data['test_label']

In [85]:
attributes

[np.str_('Female'),
 np.str_('AgeOver60'),
 np.str_('Age18-60'),
 np.str_('AgeLess18'),
 np.str_('Front'),
 np.str_('Side'),
 np.str_('Back'),
 np.str_('Hat'),
 np.str_('Glasses'),
 np.str_('HandBag'),
 np.str_('ShoulderBag'),
 np.str_('Backpack'),
 np.str_('HoldObjectsInFront'),
 np.str_('ShortSleeve'),
 np.str_('LongSleeve'),
 np.str_('UpperStride'),
 np.str_('UpperLogo'),
 np.str_('UpperPlaid'),
 np.str_('UpperSplice'),
 np.str_('LowerStripe'),
 np.str_('LowerPattern'),
 np.str_('LongCoat'),
 np.str_('Trousers'),
 np.str_('Shorts'),
 np.str_('Skirt&Dress'),
 np.str_('boots')]

In [86]:
train_images

[np.str_('000001.jpg'),
 np.str_('000002.jpg'),
 np.str_('000003.jpg'),
 np.str_('000004.jpg'),
 np.str_('000005.jpg'),
 np.str_('000006.jpg'),
 np.str_('000007.jpg'),
 np.str_('000008.jpg'),
 np.str_('000009.jpg'),
 np.str_('000010.jpg'),
 np.str_('000011.jpg'),
 np.str_('000012.jpg'),
 np.str_('000013.jpg'),
 np.str_('000014.jpg'),
 np.str_('000015.jpg'),
 np.str_('000016.jpg'),
 np.str_('000017.jpg'),
 np.str_('000018.jpg'),
 np.str_('000019.jpg'),
 np.str_('000020.jpg'),
 np.str_('000021.jpg'),
 np.str_('000022.jpg'),
 np.str_('000023.jpg'),
 np.str_('000024.jpg'),
 np.str_('000025.jpg'),
 np.str_('000026.jpg'),
 np.str_('000027.jpg'),
 np.str_('000028.jpg'),
 np.str_('000029.jpg'),
 np.str_('000030.jpg'),
 np.str_('000031.jpg'),
 np.str_('000032.jpg'),
 np.str_('000033.jpg'),
 np.str_('000034.jpg'),
 np.str_('000035.jpg'),
 np.str_('000036.jpg'),
 np.str_('000037.jpg'),
 np.str_('000038.jpg'),
 np.str_('000039.jpg'),
 np.str_('000040.jpg'),
 np.str_('000041.jpg'),
 np.str_('000042

In [87]:
class PA100kDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0])
        image = Image.open(img_name)
        targets = torch.tensor(self.annotations.iloc[idx, 1:].values.astype(float), dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, targets

In [ ]:
# Define transformations for the images
train_transforms = v2.Compose([
    v2.PILToTensor(),
    v2.Resize((299,299), antialias=True),
    v2.RandAugment(num_ops=5),
    v2.ToDtype(torch.float32),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = v2.Compose([
    v2.PILToTensor(),
    v2.Resize((299,299), antialias=True),
    v2.ToDtype(torch.float32),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


test_transforms = v2.Compose([
    v2.PILToTensor(),
    v2.Resize((299, 299), antialias=True),  # match train/val
    v2.ToDtype(torch.float32),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Path
data = '/media/tanmoy002/HDD/Pedestrian Attribute Analysis/PA-100K/data'
train_csv = '/media/tanmoy002/HDD/Pedestrian Attribute Analysis/PA-100K/train.csv'
val_csv = '/media/tanmoy002/HDD/Pedestrian Attribute Analysis/PA-100K/val.csv'
test_csv = '/media/tanmoy002/HDD/Pedestrian Attribute Analysis/PA-100K/test.csv'

# Create dataset instances
train_dataset = PA100kDataset(csv_file=train_csv, root_dir=data, transform=train_transforms)
val_dataset = PA100kDataset(csv_file=val_csv, root_dir=data, transform=val_transforms)
test_dataset = PA100kDataset(csv_file=test_csv, root_dir=data, transform=test_transforms)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [89]:
train_loader

In [90]:
train_images, train_targets = next(iter(train_loader))

In [91]:
train_images 

tensor([[[[  -2.1179,   -2.1179,   -2.1179,  ...,   -2.1179,   -2.1179,
             -2.1179],
          [  -2.1179,   -2.1179,   -2.1179,  ...,   -2.1179,   -2.1179,
             -2.1179],
          [  -2.1179,   -2.1179,   -2.1179,  ...,   -2.1179,   -2.1179,
             -2.1179],
          ...,
          [  -2.1179,   -2.1179,   -2.1179,  ...,   -2.1179,   -2.1179,
             -2.1179],
          [  -2.1179,   -2.1179,   -2.1179,  ...,   -2.1179,   -2.1179,
             -2.1179],
          [  -2.1179,   -2.1179,   -2.1179,  ...,   -2.1179,   -2.1179,
             -2.1179]],

         [[  -2.0357,   -2.0357,   -2.0357,  ...,   -2.0357,   -2.0357,
             -2.0357],
          [  -2.0357,   -2.0357,   -2.0357,  ...,   -2.0357,   -2.0357,
             -2.0357],
          [  -2.0357,   -2.0357,   -2.0357,  ...,   -2.0357,   -2.0357,
             -2.0357],
          ...,
          [  -2.0357,   -2.0357,   -2.0357,  ...,   -2.0357,   -2.0357,
             -2.0357],
          [  -2.03

In [92]:
train_images[0]

tensor([[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         ...,
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],

        [[-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         ...,
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],

        [[-1.8044, -1.8044, -1.8044,  ..., -1.8044, -1.8044, -1.8044],
         [-1.8044, -1.8044, -1.8044,  ..., -1

In [93]:
train_images[0].shape

torch.Size([3, 299, 299])

In [94]:
train_targets[0]

tensor([0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0.])

In [95]:
train_targets[0].shape

torch.Size([26])

In [96]:
train_targets.shape

torch.Size([4, 26])

In [97]:
Val_images,Val_targets=next(iter(val_loader))

In [98]:
test_images,test_loader=next(iter(test_loader))

In [99]:
import torch
print("Torch version:", torch.__version__)
print("CUDA used by torch:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.10.0+cu128
CUDA used by torch: 12.8
GPU available: True


### Training & Testing of HPnet model

In [100]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HP(num_classes=26)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 5.60 GiB of which 25.81 MiB is free. Including non-PyTorch memory, this process has 5.14 GiB memory in use. Of the allocated memory 4.99 GiB is allocated by PyTorch, and 65.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Training

In [ ]:
num_epochs = 50
best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}] Training Loss: {epoch_loss:.4f}")

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}] Validation Loss: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "HP_best.pth")
        print("Saved Best Model")

/media/tanmoy002/HDD/Pedestrian Attribute Analysis/HP_model/AF.py:39: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  att1 = F.upsample(att, scale_factor=2)
/media/tanmoy002/HDD/Pedestrian Attribute Analysis/HP_model/AF.py:44: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  att2 = F.upsample(att, scale_factor=2)
/media/tanmoy002/HDD/Pedestrian Attribute Analysis/HP_model/AF.py:45: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  att1 = F.upsample(att2, scale_factor=2)
/media/tanmoy002/HDD/Pedestrian Attribute Analysis/HP_model/AF.py:49: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  att2 = F.upsample(att, scale_factor=2)
/media/tanmoy002/HDD/Pedestrian Attribute Analysis/HP_model/AF.py:50: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  att1 = F.upsa

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 5.60 GiB of which 28.12 MiB is free. Including non-PyTorch memory, this process has 5.08 GiB memory in use. Of the allocated memory 4.91 GiB is allocated by PyTorch, and 77.64 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)